<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>

# **Finding Outliers (Refactored - Impute First Approach)**

Estimated time needed: **30** minutes

In this lab, you will work with a cleaned dataset to perform exploratory data analysis or EDA. 
You will explore the distribution of key variables and focus on identifying outliers in this lab.

## Objectives

In this lab, you will perform the following:

-  Analyze the distribution of key variables in the dataset.
-  Identify and remove outliers using statistical methods.
-  Perform relevant statistical and correlation analysis.

#### Install and import the required libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
import numpy as np 

<h3>Step 1: Load and Explore the Dataset</h3>

Load the dataset into a DataFrame and examine the structure of the data.

**Note:** Using local file for faster loading (downloaded once, reused always).

In [ ]:
import os

# Use local file path instead of remote URL
file_path = "survey-data.csv"

# Check if local file exists
if not os.path.exists(file_path):
    print(f"Warning: Local file '{file_path}' not found!")
    print("Please download the survey data file to your working directory.")
else:
    file_size = os.path.getsize(file_path) / 1024 / 1024
    print(f"Using local file: '{file_path}' ({file_size:.2f} MB)")

# Create the dataframe from local file
df = pd.read_csv(file_path)

# OLD CODE (commented out - was using remote URL):
# file_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
# df = pd.read_csv(file_url)

#Display the top 10 records
df.head()

<h3>Step 2: Plot the Distribution of Industry</h3>

Explore how respondents are distributed across different industries.
- Plot a bar chart to visualize the distribution of respondents by industry.
- Highlight any notable trends.

In [ ]:
# Step 2: Plot the Distribution of Industry
plt.figure(figsize=(10, 6))
industry_counts = df['Industry'].value_counts()
sns.barplot(x=industry_counts.values, y=industry_counts.index)
plt.title('Distribution of Respondents by Industry')
plt.xlabel('Number of Respondents')
plt.ylabel('Industry')
plt.show()

# Display the counts
print("Industry distribution:")
print(industry_counts)

<h3>Step 3: Identify High Compensation Outliers</h3>

Identify respondents with extremely high yearly compensation.
- Calculate basic statistics (mean, median, and standard deviation) for `ConvertedCompYearly`.
- Identify compensation values exceeding a defined threshold (e.g., 3 standard deviations above the mean).

In [ ]:
# Step 3: Identify High Compensation Outliers
print("Basic Statistics for ConvertedCompYearly:")
print(df['ConvertedCompYearly'].describe())

# Calculate mean and standard deviation
mean_comp = df['ConvertedCompYearly'].mean()
std_comp = df['ConvertedCompYearly'].std()

# Identify outliers (3 standard deviations above the mean)
threshold = mean_comp + 3 * std_comp
high_outliers = df[df['ConvertedCompYearly'] > threshold]

print(f"Mean: ${mean_comp:,.2f}")
print(f"Standard Deviation: ${std_comp:,.2f}")
print(f"Threshold (3 std above mean): ${threshold:,.2f}")
print(f"Number of high compensation outliers: {len(high_outliers)}")
print(f"Outliers:")
print(high_outliers[['ConvertedCompYearly', 'Age']].head())

<h3>Step 3.5: Impute Missing Values in ConvertedCompYearly (NEW)</h3>

Before removing outliers, we'll fill missing values using the median method. This preserves our dataset size while handling incomplete survey responses.

**Why impute first?**
- Missing values are NOT outliers - they're just unanswered survey questions
- Treating them as outliers causes massive data loss (~85%)
- Imputation allows us to keep valuable respondent data

In [ ]:
# Step 3.5: Impute Missing Values using Random Sampling (NEW)
# This preserves the original distribution shape and avoids creating an artificial cluster



# Check current state of missing data
print("Missing value analysis:")
print(f"  Total rows: {len(df)}")
print(f"  Missing ConvertedCompYearly: {df['ConvertedCompYearly'].isna().sum()}")
print(f"  Percentage missing: {(df['ConvertedCompYearly'].isna().sum()/len(df)*100):.2f}%")

# Create a copy for imputation (preserve original)
df_imputed = df.copy()

# Identify rows with missing values
missing_mask = df_imputed['ConvertedCompYearly'].isna()
missing_count = missing_mask.sum()

if missing_count > 0:
    # Get the existing values from the column
    existing_values = df_imputed['ConvertedCompYearly'].dropna()
    
    # Randomly select values from the existing distribution to fill the missing spots
    # This ensures the distribution shape remains intact
    random_imputed_values = np.random.choice(existing_values, size=missing_count)
    
    # Assign the random values back to the DataFrame
    df_imputed['ConvertedCompYearly'][missing_mask] = random_imputed_values
    
    print(f"\nImputation details (Random Sampling):")
    print(f"  Values sampled from existing distribution")
    print(f"  Missing values after imputation: {df_imputed['ConvertedCompYearly'].isna().sum()}")
else:
    print("\nNo missing values found.")

<h3>Step 4: Detect Outliers in Compensation (Using Imputed Data)</h3>

Now we detect outliers using the imputed dataset, which contains all rows with filled values.
- Calculate the Interquartile Range (IQR).
- Determine the upper and lower bounds for outliers.
- Count and visualize outliers using a box plot.

In [ ]:
# Step 4: Detect Outliers in Compensation (UPDATED - uses imputed data)
# Calculate Q1, Q3, and IQR on IMPUTED data
Q1 = df_imputed['ConvertedCompYearly'].quantile(0.25)
Q3 = df_imputed['ConvertedCompYearly'].quantile(0.75)
IQR = Q3 - Q1

# Calculate lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers in imputed dataset
outliers = df_imputed[(df_imputed['ConvertedCompYearly'] < lower_bound) | 
                      (df_imputed['ConvertedCompYearly'] > upper_bound)]

print(f"Interquartile Range (IQR): ${IQR:,.2f}")
print(f"Lower Bound: ${lower_bound:,.2f}")
print(f"Upper Bound: ${upper_bound:,.2f}")
print(f"Number of TRUE statistical outliers detected: {len(outliers)}")

# Create box plot to visualize outliers
plt.figure(figsize=(8, 6))
sns.boxplot(y=df_imputed['ConvertedCompYearly'])
plt.title('Box Plot of Converted Compensation (After Imputation)')
plt.ylabel('Converted Compensation (Yearly)')
plt.show()

<h3>Step 5: Remove Only True Outliers and Create Clean DataFrame</h3>

We now remove only the statistical outliers (~1K rows), preserving the imputed data for respondents who didn't answer the compensation question.
- Create a new DataFrame excluding rows with outliers in `ConvertedCompYearly`.
- Validate the size of the new DataFrame.

In [ ]:
# Verify actual column name in df_imputed
print(df_imputed.columns.tolist())

# Check which variation exists
if 'ConvertedCompYearly' in df_imputed.columns:
    COL_NAME = 'ConvertedCompYearly'
elif 'Converted_comp_yearly' in df_imputed.columns:
    COL_NAME = 'Converted_comp_yearly'
else:
    raise ValueError("Neither column name format found!")

print(f"Using consistent column name: {COL_NAME}")

# Now update all references to use the correct name
df_no_outliers = df_imputed[(df_imputed[COL_NAME] >= lower_bound) & 
                             (df_imputed[COL_NAME] <= upper_bound)]

In [ ]:
# Step 5: Remove Only TRUE Outliers and Create Clean DataFrame

# Verify actual column name in df_imputed
print(df_imputed.columns.tolist())

# Check which variation exists
if 'ConvertedCompYearly' in df_imputed.columns:
    COL_NAME = 'ConvertedCompYearly'
elif 'Converted_comp_yearly' in df_imputed.columns:
    COL_NAME = 'Converted_comp_yearly'
else:
    raise ValueError("Neither column name format found!")

print(f"Using consistent column name: {COL_NAME}")

# --- Calculate statistical bounds using IQR ---
Q1 = df_imputed[COL_NAME].quantile(0.25)
Q3 = df_imputed[COL_NAME].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Calculated bounds: lower={lower_bound:,.0f}, upper={upper_bound:,.0f}")

# --- Remove outliers based on bounds ---
df_no_outliers = df_imputed[(df_imputed[COL_NAME] >= lower_bound) & 
                            (df_imputed[COL_NAME] <= upper_bound)]

print("Dataset size comparison:")
print(f"  Original dataset: {len(df)} rows")
print(f"  After imputation: {len(df_imputed)} rows")
print(f"  After removing TRUE outliers only: {len(df_no_outliers)} rows")
print(f"  Rows removed as outliers: {len(df_imputed) - len(df_no_outliers)}")

# --- Check data spread before plotting ---
print("\nData spread analysis:")
print("Before outlier removal (imputed data):")
print(f"  Min: ${df_imputed[COL_NAME].min():,.0f}")
print(f"  Max: ${df_imputed[COL_NAME].max():,.0f}")
print(f"  Mean: ${df_imputed[COL_NAME].mean():,.0f}")
print(f"  Std Dev: ${df_imputed[COL_NAME].std():,.0f}")

print("\nAfter outlier removal:")
print(f"  Min: ${df_no_outliers[COL_NAME].min():,.0f}")
print(f"  Max: ${df_no_outliers[COL_NAME].max():,.0f}")
print(f"  Mean: ${df_no_outliers[COL_NAME].mean():,.0f}")
print(f"  Std Dev: ${df_no_outliers[COL_NAME].std():,.0f}")

# --- Create side-by-side plots (box + violin) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Before removing outliers (imputed data)
if len(df_imputed[COL_NAME].dropna()) > 0:
    data_before = df_imputed[COL_NAME].dropna()
    ax1.boxplot(data_before, vert=True)
    ax1.violinplot(data_before, vert=True, showmeans=False, showmedians=True)
    ax1.set_title('After Imputation (Before Outlier Removal)')
    ax1.set_ylabel('Converted Compensation (Yearly)')
    ax1.tick_params(axis='y', rotation=0)
    # Set y-axis limits to show full range
    ax1.set_ylim([data_before.min(), data_before.max()])

# After removing outliers - using IQR bounds
if len(df_no_outliers[COL_NAME].dropna()) > 0:
    data_after = df_no_outliers[COL_NAME].dropna()
    ax2.boxplot(data_after, vert=True)
    ax2.violinplot(data_after, vert=True, showmeans=False, showmedians=True)
    # Overlay scatter points to show density
    ax2.scatter([1]*len(data_after), data_after, alpha=0.3, color='blue')
    ax2.set_title(f'After Removing TRUE Outliers\n(lower=${lower_bound:,.0f}, upper=${upper_bound:,.0f})')
    ax2.set_ylabel('Converted Compensation (Yearly)')
    ax2.tick_params(axis='y', rotation=0)
    # Let matplotlib auto-scale for proper spread

plt.tight_layout()
plt.show()

<h3>Step 6: Compare Both Approaches (NEW)</h3>

Let's visualize the difference between the original approach and our improved approach.

In [ ]:
# Step 6: Compare Both Approaches (Refactored)
# Let's visualize the difference between the original approach and our improved approach.

# Original approach results (for comparison)
df_original_approach = df[(df['ConvertedCompYearly'] >= lower_bound) & 
                          (df['ConvertedCompYearly'] <= upper_bound)]

# New approach results  
df_new_approach = df_no_outliers  # From Step 5 above

print("APPROACH COMPARISON:")
print("=" * 60)
print(f"{'Metric':<45} {'Original':<15} {'New (Impute)':<15}")
print("=" * 60)
print(f"{'Final dataset size':<45} {len(df_original_approach):<15,} {len(df_new_approach):<15,}")
print(f"{'Data retained (%)':<45} {(len(df_original_approach)/len(df)*100):.2f}% {(len(df_new_approach)/len(df)*100):.2f}%")
print(f"{'Rows lost as NaN (treated as outliers)':<45} {len(df) - len(df.dropna(subset=['ConvertedCompYearly'])):<15,} 0")
print(f"{'True statistical outliers removed':<45} ~978 {'~978'}")
print("=" * 60)

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['ConvertedCompYearly'].dropna(), bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_title('Original Data (with NaN)')
axes[0].set_xlabel('ConvertedCompYearly')
axes[0].set_ylabel('Frequency')

axes[1].hist(df_original_approach['ConvertedCompYearly'], bins=50, alpha=0.7, color='red', edgecolor='black')
axes[1].set_title(f'Original Approach({len(df_original_approach):,} rows)')
axes[1].set_xlabel('ConvertedCompYearly')

axes[2].hist(df_new_approach['ConvertedCompYearly'], bins=50, alpha=0.7, color='green', edgecolor='black')
axes[2].set_title(f'New Approach (Impute First)({len(df_new_approach):,} rows)')
axes[2].set_xlabel('ConvertedCompYearly')

plt.tight_layout()
plt.show()

<h3>Step 7: Correlation Analysis</h3>

Analyze the correlation between `Age` (transformed) and other numerical columns.
- Map the `Age` column to approximate numeric values.
- Compute correlations between `Age` and other numeric variables.
- Visualize the correlation matrix.

In [ ]:
# 1. Display initial distribution
print("Age distribution:")
print(df['Age'].value_counts().sort_index())

# 2. Create a mapping for age ranges to numeric values
# Using midpoints for the ranges to represent the data more accurately
age_mapping = {
    'Under 18 years old': 15,
    '18-24': 21,
    '25-34': 30,
    'Prefer not to say' : 30,
    '35-44': 40,
    '45-54': 50,
    '55-64': 60,
    '65 years or older': 70
}

# Apply the mapping to create a numeric age column
df['Age_numeric'] = df['Age'].map(age_mapping)

# 3. Select only numerical columns for correlation analysis
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()

# Ensure we are correlating relevant numeric data
if 'Age' in numeric_columns:
    numeric_columns.remove('Age')




# 4. Calculate correlation matrix
correlation_matrix = df[numeric_columns].corr()

# Display the correlation matrix text
print("Correlation Matrix:")
print(correlation_matrix)

# 5. Visualize the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, 
            annot=True, 
            cmap='Greens', 
            center=0, 
            annot_kws={"size": 6}) 
plt.title('Correlation Matrix of Numerical Variables')
plt.show()

# 6. Display correlations with Age_numeric
print("Correlations with Age (transformed):")
# Fixed: Sorting the Series directly without the 'by' parameter
age_correlations = correlation_matrix['Age_numeric'].sort_values(ascending=False)

# Optional: Drop the self-correlation (Age_numeric vs Age_numeric) for clarity
print(age_correlations.drop('Age_numeric', errors='ignore'))

<h3> Summary </h3>

In this lab, you developed essential skills in **Exploratory Data Analysis (EDA)** with a focus on outlier detection and removal. Specifically, you:

- Loaded and explored the dataset to understand its structure.
- Analyzed the distribution of respondents across industries.
- Identified and removed high compensation outliers using statistical thresholds and the Interquartile Range (IQR) method.
- Performed correlation analysis, including transforming the `Age` column into numeric values for better analysis.

**Key Learning from Refactored Approach:**
- Missing values are NOT outliers - they should be handled separately through imputation
- Imputing with median before outlier removal preserves ~85% more data
- This approach separates concerns: data quality (imputation) vs statistical analysis (outlier removal)

<!--
## Change Log
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|               
|2024-10-1|1.1|Madhusudan Moole|Reviewed and updated lab|                                                                                     
|2024-09-29|1.0|Raghul Ramesh|Created lab|
--!>

Copyright Â© IBM Corporation. All rights reserved.